## Ensemble testes
Esse script implementa um ensemble por média entre dois modelos de Gradient Boosting:
* LightGBM (rodando em GPU, com categorias nativas)
* XGBoost (rodando em CPU, com árvores histogram)
* Catbost

## 1.Bibliotecas

In [1]:
import sys
import warnings
import os
import pandas as pd
import numpy as np
import joblib
import time
from pathlib import Path

# Modelagem e Métricas
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from sklearn.linear_model import LogisticRegression

# Importações locais
from pathlib import Path
from setup_notebook import setup_path
setup_path()
from src.model_utils import *
from src.plot_metrica_class import *
from src.preprocess_utils_diab3a import * #(NOVO atualizações)


# from src.plot_metrica_class import * warnings.filterwarnings("ignore")
print(f"# Processo iniciado em: {time.strftime('%H:%M:%S')}")
start_inicial = time.time()

# Processo iniciado em: 18:26:55


## 2-DataLoad

In [2]:
# Definindo caminhos de pastas
BASE = Path.cwd().parent   
DATA_DIR = BASE / "data" / "raw"
DATA_MODELS = BASE / "models" 


#1. Carregamento dos dados
#forma 1: usa todos os dados
dfo = pd.read_csv("/home/akel/PycharmProjects/Kaggle/Diabetes_Prediction_Challenge/data/raw/train.csv")
df = dfo.drop(columns='id')
TARGET = "diagnosed_diabetes"
X_train = df.drop(columns=TARGET)
y_train = df[TARGET]


# Dados de Teste (Kaggle)
df_test = pd.read_csv(DATA_DIR / "test.csv")
id_test = df_test["id"]
X_test_final = df_test.drop(columns='id')

# 2. Carregamento dos modelos pré-treinados
# Nota: Esses modelos geralmente são Pipelines que já incluem o preprocessador
pipe_XGB = joblib.load(DATA_MODELS / 'modelo_XGB2_final_randsearch.roc_auc_v3.joblib') # AUC Score : 0.7272
pipe_CBT = joblib.load(DATA_MODELS / 'modelo_CBT_final_base.roc_auc_v3a.joblib') # AUC Score : 0.7252
search_h = joblib.load(DATA_MODELS /'search_Hrandomearly_stop_v3a.joblib') # AUC Score : 0.7255
pipe_LGBM= search_h.best_estimator_

print("✅ Dados e modelos carregados com sucesso!")

✅ Dados e modelos carregados com sucesso!


In [3]:
print(f"# Processo iniciado em: {time.strftime('%H:%M:%S')}")

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays vazios para guardar as probabilidades de cada modelo
oof_lgbm = np.zeros(len(X_train))
oof_xgb  = np.zeros(len(X_train))
oof_cbt  = np.zeros(len(X_train))


print(f"🚀 Iniciando Cross-Validation ({N_SPLITS} folds)...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    T_i_fold = time.time()

    # --- LGBM ---
    start_m = time.time()
    pipe_LGBM.fit(X_train.iloc[train_idx], y_train[train_idx])
    print(f"    ✅ LGBM - ⏱️ {time.time() - start_m:5.2f}s")
    
    # --- XGB  ---
    start_m = time.time()
    pipe_XGB.fit(X_train.iloc[train_idx], y_train[train_idx])
    print(f"    ✅ XGB  - ⏱️ {time.time() - start_m:5.2f}s")

    
    # --- CBT  ---
    start_m = time.time()
    pipe_CBT.fit(X_train.iloc[train_idx], y_train[train_idx],model__logging_level='Silent')
    print(f"    ✅ CBT  - ⏱️ {time.time() - start_m:5.2f}s")

    
    # Predict_proba nos dados de validação do fold
    oof_lgbm[val_idx] = pipe_LGBM.predict_proba(X_train.iloc[val_idx])[:, 1]
    oof_xgb[val_idx]  = pipe_XGB.predict_proba(X_train.iloc[val_idx])[:, 1]
    oof_cbt[val_idx]  = pipe_CBT.predict_proba(X_train.iloc[val_idx])[:, 1]

    print(f"Fold {fold+1} concluído. Tempo: {(time.time()-T_i_fold):3.2f}s ")

# Cálculo do score base de cada um
print(f"\nScore Individual LGBM: {roc_auc_score(y_train, oof_lgbm):.4f}")
print(f"Score Individual XGB:  {roc_auc_score(y_train, oof_xgb):.4f}")
print(f"Score Individual CBT:  {roc_auc_score(y_train, oof_cbt):.4f}")
print(f"# Processo finalizado em: {time.strftime('%H:%M:%S')}")
#Score Individual LGBM: 0.7293
#Score Individual XGB:  0.7277
#------------------
#Score Individual LGBM: 0.7293
#Score Individual XGB:  0.7284
#~7min


#Score Individual LGBM: 0.7293
#Score Individual XGB:  0.7284
#Score Individual CBT:  0.7265
#41 min

# Processo iniciado em: 18:26:57
🚀 Iniciando Cross-Validation (5 folds)...
    ✅ LGBM - ⏱️ 32.31s
    ✅ XGB  - ⏱️ 37.79s
    ✅ CBT  - ⏱️ 296.17s
Fold 1 concluído. Tempo: 375.01s 
    ✅ LGBM - ⏱️ 32.60s
    ✅ XGB  - ⏱️ 37.98s
    ✅ CBT  - ⏱️ 296.23s
Fold 2 concluído. Tempo: 375.87s 
    ✅ LGBM - ⏱️ 32.49s
    ✅ XGB  - ⏱️ 38.61s
    ✅ CBT  - ⏱️ 297.25s
Fold 3 concluído. Tempo: 377.54s 
    ✅ LGBM - ⏱️ 32.70s
    ✅ XGB  - ⏱️ 37.49s
    ✅ CBT  - ⏱️ 294.94s
Fold 4 concluído. Tempo: 374.61s 
    ✅ LGBM - ⏱️ 32.66s
    ✅ XGB  - ⏱️ 37.46s
    ✅ CBT  - ⏱️ 296.93s
Fold 5 concluído. Tempo: 376.65s 

Score Individual LGBM: 0.7293
Score Individual XGB:  0.7284
Score Individual CBT:  0.7265
# Processo finalizado em: 18:58:17


In [4]:
# Correlação dos Modelos.
oof_df = pd.DataFrame({
    'lgbm': oof_lgbm,
    'xgb': oof_xgb,
    'cbt': oof_cbt
})
print("\n📊 Correlação entre as predições OOF:")
print(oof_df.corr())


📊 Correlação entre as predições OOF:
          lgbm       xgb       cbt
lgbm  1.000000  0.983588  0.975768
xgb   0.983588  1.000000  0.986232
cbt   0.975768  0.986232  1.000000


In [5]:
# Treino final com 100% dos dados de treino
print("treino com 100% dos dados")
pipe_LGBM.fit(X_train, y_train)
print(f"✅ LGBM finalizado • {time.strftime('%H:%M:%S')}")

pipe_XGB.fit(X_train, y_train)
print(f"✅ XGB finalizado  • {time.strftime('%H:%M:%S')}")
pipe_CBT.fit(X_train, y_train)
print(f"✅ CBT finalizado  • {time.strftime('%H:%M:%S')}")
# Predição na base de teste real
p_lgbm = pipe_LGBM.predict_proba(X_test_final)[:, 1]
p_xgb  = pipe_XGB.predict_proba(X_test_final)[:, 1]
p_cbt  = pipe_CBT.predict_proba(X_test_final)[:, 1]


treino com 100% dos dados
✅ LGBM finalizado • 18:58:57
✅ XGB finalizado  • 18:59:44
Learning rate set to 0.168963
0:	total: 301ms	remaining: 5m
999:	total: 6m 8s	remaining: 0us
✅ CBT finalizado  • 19:05:55


In [6]:
thresholds = np.linspace(0.01, 0.99, 99)

oof_ensemble_base = (oof_lgbm + oof_xgb + oof_cbt) / 3 # Media simples para teste
f1_scores = [f1_score(y_train, (oof_ensemble_base >= t).astype(int)) for t in thresholds]

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

# Alternativa: Índice de Youden (Equilíbrio entre Sensibilidade e Especificidade)
fpr, tpr, thresh_roc = roc_curve(y_train, oof_ensemble_base)
youden_threshold = thresh_roc[np.argmax(tpr - fpr)]

print(f"🎯 Melhor threshold (F1): {best_threshold:.3f} com F1 de {max(f1_scores):.4f}")
print(f"🎯 Threshold (Youden): {youden_threshold:.3f}")
# 🎯 Melhor threshold (F1): 0.400 com F1 de 0.7810
# 🎯 Threshold (Youden): 0.626

🎯 Melhor threshold (F1): 0.410 com F1 de 0.7813
🎯 Threshold (Youden): 0.618


In [7]:
# 1) Aplicando pesos de Blending (ajuste conforme peso manualmente)
p_ensemble_final = 0.45 * p_lgbm + 0.35 * p_xgb + 0.20*p_cbt
# Aplicando o threshold escolhido 
y_pred_binario = (p_ensemble_final >= youden_threshold).astype(int)

# Gerando CSV
submission = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred_binario
})

submission.to_csv("submission_ensemble_Blending_manual_3model_v3.csv", index=False)
print(f"✅ Arquivo de submissão salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")

✅ Arquivo de submissão salvo • 05/03/2026-19:06:31


In [9]:
# 2) calculando os pesos via regressão logisca
# Criamos uma nova matriz onde as colunas são as previsões do LGBM e XGB

X_meta = np.column_stack([oof_lgbm, oof_xgb,oof_cbt])
X_meta_test = np.column_stack([p_lgbm, p_xgb,p_cbt])

# Meta-modelo (Regressão Logística é o padrão para Stacking)
meta_model = LogisticRegression(random_state=42)
meta_model.fit(X_meta, y_train)

# Score do Stacking 
p_meta_oof = meta_model.predict_proba(X_meta)[:, 1]
print(f"🚀 ROC AUC do Stacking: {roc_auc_score(y_train, p_meta_oof):.4f}")


# 2. Usar o meta-modelo para prever as probabilidades finais
# O meta_model aplica os pesos e o intercepto automaticamente
p_stacking_final = meta_model.predict_proba(X_meta_test)[:, 1] 

fpr, tpr, thresholds_meta = roc_curve(y_train, p_meta_oof)
youden_threshold_stk = thresholds_meta[np.argmax(tpr - fpr)]

print(f"🎯 Novo Threshold Otimizado para Stacking: {youden_threshold_stk:.3f}")


y_pred_stacking = (p_stacking_final >= youden_threshold_stk).astype(int)


# 4. Criar o DataFrame de submissão
submission_stacking = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred_stacking
})

submission_stacking.to_csv("submission_ensemble_stacking_3models_v3.csv", index=False)
print(f"✅ Submissão Stacking salva • {time.strftime('%d/%m/%Y-%H:%M:%S')}")
# # Acessando os coeficientes (pesos) do meta-modelo
pesos = meta_model.coef_[0]
intercept = meta_model.intercept_[0]

# print(f"⚖️ Peso atribuído ao LGBM: {pesos[0]:.4f}")
# print(f"⚖️ Peso atribuído ao XGB:  {pesos[1]:.4f}")
# print(f"📌 Intercepto (Viés):     {intercept:.4f}")

# Para transformar em pesos percentuais (aproximados)
soma_pesos = np.abs(pesos).sum()
print(f"\nImportância Relativa (estimada):")
print(f"LGBM: {(np.abs(pesos[0])/soma_pesos)*100:.1f}%")
print(f"XGB:  {(np.abs(pesos[1])/soma_pesos)*100:.1f}%")
print(f"CBT:  {(np.abs(pesos[2])/soma_pesos)*100:.1f}%")



🚀 ROC AUC do Stacking: 0.7298
🎯 Novo Threshold Otimizado para Stacking: 0.633
✅ Submissão Stacking salva • 05/03/2026-19:06:33

Importância Relativa (estimada):
LGBM: 57.9%
XGB:  29.4%
CBT:  12.7%


In [10]:
#) Otimização Numérica
from scipy.optimize import minimize

# Função que queremos MINIMIZAR (negativo do AUC)
def objective(weights):
    # Garante que as probabilidades combinadas fiquem entre 0 e 1
    p_ensemble = weights[0] * oof_lgbm + weights[1] * oof_xgb+ weights[2] * oof_cbt
    return -roc_auc_score(y_train, p_ensemble)

# Chute inicial (50/50)
v_inicial = [0.4, 0.4, 0.2]
# Restrições: pesos entre 0 e 1; a soma deve ser 1.0
bounds = [(0, 1), (0, 1),(0,1)]
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
res = minimize(objective, v_inicial, bounds=bounds, constraints=constraints)
w1, w2, w3 = res.x
print(f"🏆 Pesos Ótimos: LGBM = {w1:.4f}, XGB = {w2:.4f}, CBT = {w3:.4f}")
print(f"⭐ Melhor AUC possível via Blending: {-res.fun:.4f}")



#1) Aplicando pesos calculados no treino
p_oof_otmz = w1 * oof_lgbm + w2 * oof_xgb + w3 * oof_cbt

fpr, tpr, thresh_roc = roc_curve(y_train, p_oof_otmz)
youden_threshold_otmz = thresh_roc[np.argmax(tpr - fpr)]

print(f"🎯 Threshold Otimizado (Youden): {youden_threshold_otmz:.4f}")

# 3) Aplicar os pesos nos dados de TESTE (Submissão)
p_test_otimizado = w1 * p_lgbm + w2 * p_xgb+ w3 * p_cbt

# 4) Aplicar o threshold calibrado nas predições de teste
y_pred_otmz = (p_test_otimizado >= youden_threshold_otmz).astype(int)

# 5) Gerando CSV
submission = pd.DataFrame({
    "id": id_test,
    "Survived": y_pred_otmz
})

submission.to_csv("submission_ensemble_objet_3model_v3.csv", index=False)
print(f"✅ Arquivo de submissão salvo • {time.strftime('%d/%m/%Y-%H:%M:%S')}")

🏆 Pesos Ótimos: LGBM = 0.5634, XGB = 0.3080, CBT = 0.1286
⭐ Melhor AUC possível via Blending: 0.7298
🎯 Threshold Otimizado (Youden): 0.6154
✅ Arquivo de submissão salvo • 05/03/2026-19:06:38
